# Μάθημα 11 - Model Context Protocol (MCP)

Το **Model Context Protocol (MCP)** είναι ένα ανοιχτό πρότυπο που επιτρέπει στους πράκτορες να ανακαλύπτουν δυναμικά και να χρησιμοποιούν εργαλεία, πόρους και πηγές δεδομένων κατά την εκτέλεση. Αντί να ενσωματώνονται με σταθερό τρόπο εργαλεία σε έναν πράκτορα, το MCP επιτρέπει στους πράκτορες να συνδέονται με εξωτερικούς διακομιστές που εκθέτουν δυνατότητες κατ' απαίτηση.

Σε αυτό το μάθημα, θα μάθετε:
- Τι είναι το MCP και γιατί έχει σημασία για τα συστήματα πρακτόρων
- Πώς λειτουργεί η αρχιτεκτονική πελάτη-διακομιστή του MCP
- Πώς να κατασκευάσετε πράκτορες που χρησιμοποιούν ανακάλυψη εργαλείων με στυλ MCP


## Ρύθμιση

**Προαπαιτούμενα:**
- Έργο Azure AI Foundry με αναπτυγμένο μοντέλο
- Εκτελέστε `az login` για έλεγχο ταυτότητας


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Τι είναι το Model Context Protocol (MCP);

Το MCP ορίζει έναν τυποποιημένο τρόπο για πράκτορες τεχνητής νοημοσύνης να εντοπίζουν και να αλληλεπιδρούν με εξωτερικά εργαλεία και πηγές δεδομένων:

- **MCP Server**: Εκθέτει εργαλεία, πόρους και προτροπές μέσω ενός τυποποιημένου πρωτοκόλλου
- **MCP Client**: Το περιβάλλον εκτέλεσης του πράκτορα που συνδέεται με διακομιστές και ανακαλύπτει τις διαθέσιμες δυνατότητες
- **Dynamic Discovery**: Οι πράκτορες δεν χρειάζονται ενσωματωμένα εργαλεία — ανακαλύπτουν τι είναι διαθέσιμο κατά το χρόνο εκτέλεσης

Αυτό είναι ιδιαίτερα χρήσιμο για την κατασκευή επεκτάσιμων συστημάτων πρακτόρων όπου νέες δυνατότητες μπορούν να προστεθούν χωρίς να τροποποιηθεί ο κώδικας του πράκτορα.


## Πώς λειτουργεί το MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Ο πράκτορας (πελάτης MCP) συνδέεται σε έναν διακομιστή MCP
2. Ο διακομιστής απαντά με μια λίστα διαθέσιμων εργαλείων και τα σχήματά τους
3. Ο πράκτορας μπορεί στη συνέχεια να καλέσει οποιοδήποτε εντοπισμένο εργαλείο κατά τη διάρκεια της συλλογιστικής του
4. Τα αποτελέσματα επιστρέφουν μέσω του ίδιου πρωτοκόλλου


## Προσομοίωση Ανακάλυψης Εργαλείων MCP

Δεδομένου ότι ένας πραγματικός διακομιστής MCP απαιτεί μια ενεργή διεργασία διακομιστή, θα δείξουμε το μοτίβο χρησιμοποιώντας συναρτήσεις `@tool` που εξομοιώνουν όσα θα παρείχε μια υπηρεσία καταλύματος συνδεδεμένη με MCP.

Σε παραγωγικό περιβάλλον, αυτά τα εργαλεία θα ανακαλύπτονταν δυναμικά από έναν διακομιστή MCP αντί να ορίζονται τοπικά.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Κατασκευή Πράκτορα με Εργαλεία τύπου MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP σε παραγωγή

Σε ένα περιβάλλον παραγωγής, το MCP επιτρέπει ισχυρά πρότυπα:

- **Δυναμική ανακάλυψη εργαλείων**: Οι πράκτορες συνδέονται σε διακομιστές MCP και εντοπίζουν εργαλεία κατά την εκτέλεση
- **Αποσυνδεδεμένη αρχιτεκτονική**: Οι πάροχοι εργαλείων μπορούν να ενημερώνονται ανεξάρτητα από τον πράκτορα
- **Δια-οργανωτική κοινή χρήση**: Οι ομάδες μπορούν να εκθέτουν δυνατότητες μέσω διακομιστών MCP που οποιοσδήποτε πράκτορας μπορεί να χρησιμοποιήσει
- **Υποστήριξη Microsoft Agent Framework**: Το MAF έχει ενσωματωμένη υποστήριξη πελάτη MCP μέσω της ενσωμάτωσης `mcp`

Για να χρησιμοποιήσετε έναν πραγματικό διακομιστή MCP με το MAF, θα συνδεθείτε μέσω του `hosted_mcp_tool()` ή της ενσωμάτωσης πελάτη MCP.

**Μάθετε περισσότερα:**
- [Προδιαγραφή MCP](https://modelcontextprotocol.io/)
- [Υποστήριξη MCP για Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Περίληψη

Σε αυτό το μάθημα, μάθατε:
- **MCP** είναι ένα ανοιχτό πρότυπο για τη δυναμική ανακάλυψη εργαλείων μεταξύ πρακτόρων και παρόχων εργαλείων
- Η **αρχιτεκτονική πελάτη-εξυπηρετητή** επιτρέπει στους πράκτορες να ανακαλύπτουν δυνατότητες κατά το χρόνο εκτέλεσης
- MCP επιτρέπει **επεκτάσιμα, αποσυνδεδεμένα συστήματα πρακτόρων** όπου τα εργαλεία μπορούν να προστεθούν χωρίς αλλαγές στον κώδικα
- Microsoft Agent Framework παρέχει **ενσωματωμένη υποστήριξη MCP** για χρήση στην παραγωγή


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που επιδιώκουμε την ακρίβεια, λάβετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν σφάλματα ή ανακρίβειες. Το αρχικό έγγραφο στη γλώσσα του πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
